In [1]:
import shutil, os, sys

# Copy code files
CODE = "/kaggle/input/datasets/jerinanan/pgcn-eeg-code"
for f in os.listdir(CODE):
    if f.endswith(".py"):
        shutil.copy(f"{CODE}/{f}", f"/kaggle/working/{f}")
sys.path.insert(0, "/kaggle/working")

# Correct paths based on your folder structure
EEG    = "/kaggle/input/datasets/jerinanan/pgcn-all-results/all_results/results_eeg"
FUSION = "/kaggle/input/datasets/jerinanan/pgcn-all-results/all_results/results_fusion"
FIGS   = "/kaggle/working/figures"

os.makedirs(FIGS, exist_ok=True)

ret = os.system(
    f"python /kaggle/working/pgcn_statistics_plots.py "
    f"--results_eeg {EEG} "           # ← space before --
    f"--results_fusion {FUSION} "     # ← space before --
    f"--out_dir {FIGS}"
)
print(f"✔ Done (ret={ret})")

Loading results …
  ✔ EEG-Only (Within)               (32 subjects)
  ✔ EEG-Only (LOSO)                 (32 subjects)
  ✔ Early Fusion (Within)           (22 subjects)
  ✔ Early Fusion (LOSO)             (22 subjects)
  ✔ Late Fusion (Within)            (22 subjects)
  ✔ Late Fusion (LOSO)              (22 subjects)

──────────────────────────────────────────────────────────────────────
  WILCOXON TESTS  (Within)
──────────────────────────────────────────────────────────────────────
  Valence       EEG-Only (Within)         vs Early Fusion (Within)      Δ=-0.1113  p=0.0680  ns
  Valence       EEG-Only (Within)         vs Late Fusion (Within)       Δ=-0.1118  p=0.1762  ns
  Valence       Early Fusion (Within)     vs Late Fusion (Within)       Δ=-0.0005  p=0.7439  ns
  Arousal       EEG-Only (Within)         vs Early Fusion (Within)      Δ=+0.0018  p=0.7782  ns
  Arousal       EEG-Only (Within)         vs Late Fusion (Within)       Δ=-0.0815  p=0.0674  ns
  Arousal       Early Fusion (Wi

In [2]:
import shutil

shutil.make_archive(
    "/kaggle/working/figures",  # output zip name (without .zip)
    "zip",
    "/kaggle/working/figures"
)

print("ZIP created:", "/kaggle/working/figures.zip")

ZIP created: /kaggle/working/figures.zip


In [7]:
import os, json, numpy as np

def load_results(path, is_loso=False):
    results = {}
    if not os.path.exists(path):
        print(f"  ✗ Not found: {path}")
        return results
    prefix = "fold_" if is_loso else "s"
    for fd in sorted(os.scandir(path), key=lambda x: x.name):
        rpath = f"{fd.path}/metrics.json"
        if fd.name.startswith(prefix) and os.path.exists(rpath):
            key = fd.name.replace("fold_","")
            results[key] = json.load(open(rpath))
    return results

# ── Correct base paths based on your folder structure ──
BASE_EEG    = "/kaggle/input/datasets/jerinanan/pgcn-all-results/all_results/results_eeg"
BASE_FUSION = "/kaggle/input/datasets/jerinanan/pgcn-all-results/all_results/results_fusion"

# Verify paths exist
print("Checking paths:")
print(f"  EEG    exists: {os.path.exists(BASE_EEG)}")
print(f"  FUSION exists: {os.path.exists(BASE_FUSION)}")
if os.path.exists(BASE_EEG):
    print(f"  EEG contents: {os.listdir(BASE_EEG)}")
if os.path.exists(BASE_FUSION):
    print(f"  FUSION contents: {os.listdir(BASE_FUSION)}")

Checking paths:
  EEG    exists: True
  FUSION exists: True
  EEG contents: ['loso_no_hjorth', 'loso_flat_gcn', 'loso_no_band', 'within_struct_only', 'within_none', 'within_flat_gcn', 'within_no_band_attn', 'loso_struct_only', 'loso_none', 'within_no_hjorth']
  FUSION contents: ['loso_late', 'within_late', 'loso_early', 'within_early']


In [4]:
experiments = {
    "EEG-Only Within (n=32)":     load_results(f"{BASE_EEG}/within_none"),
    "EEG-Only LOSO (n=32)":       load_results(f"{BASE_EEG}/loso_none",        True),
    "Early Fusion Within (n=22)": load_results(f"{BASE_FUSION}/within_early"),
    "Late Fusion Within (n=22)":  load_results(f"{BASE_FUSION}/within_late"),
    "Early Fusion LOSO (n=22)":   load_results(f"{BASE_FUSION}/loso_early",     True),
    "Late Fusion LOSO (n=22)":    load_results(f"{BASE_FUSION}/loso_late",      True),
    "A1 Flat GCN Within":         load_results(f"{BASE_EEG}/within_flat_gcn"),
    "A2 No Band Attn Within":     load_results(f"{BASE_EEG}/within_no_band_attn"),
    "A3 Struct Only Within":      load_results(f"{BASE_EEG}/within_struct_only"),
    "A4 No Hjorth Within":        load_results(f"{BASE_EEG}/within_no_hjorth"),
    "A1 Flat GCN LOSO":           load_results(f"{BASE_EEG}/loso_flat_gcn",     True),
    "A2 No Band Attn LOSO":       load_results(f"{BASE_EEG}/loso_no_band", True),
    "A3 Struct Only LOSO":        load_results(f"{BASE_EEG}/loso_struct_only",  True),
    "A4 No Hjorth LOSO":          load_results(f"{BASE_EEG}/loso_no_hjorth",    True),
}

print(f"\n{'Model':<30} {'N':>3} {'Valence':>8} {'Arousal':>8} "
      f"{'Dominance':>10} {'Liking':>8} {'Quadrant':>10}")
print("─"*82)

for name, res in experiments.items():
    if not res:
        print(f"{name:<30} {'NO DATA':>3}")
        continue
    v = np.mean([r["val_f1"]  for r in res.values()])
    a = np.mean([r["aro_f1"]  for r in res.values()])
    d = np.mean([r["dom_f1"]  for r in res.values()])
    l = np.mean([r["lik_f1"]  for r in res.values()])
    q = np.mean([r["quad_f1"] for r in res.values()])
    n = len(res)
    print(f"{name:<30} {n:>3} {v:>8.4f} {a:>8.4f} "
          f"{d:>10.4f} {l:>8.4f} {q:>10.4f}")


Model                            N  Valence  Arousal  Dominance   Liking   Quadrant
──────────────────────────────────────────────────────────────────────────────────
EEG-Only Within (n=32)          32   0.6189   0.6669     0.6652   0.7300     0.2821
EEG-Only LOSO (n=32)            32   0.6970   0.7045     0.7430   0.7927     0.1469
Early Fusion Within (n=22)      22   0.7081   0.6561     0.6195   0.7950     0.4143
Late Fusion Within (n=22)       22   0.7087   0.7394     0.5833   0.7969     0.4476
Early Fusion LOSO (n=22)        22   0.5034   0.5702     0.5400   0.7351     0.1701
Late Fusion LOSO (n=22)         22   0.5187   0.6120     0.5320   0.6769     0.1704
A1 Flat GCN Within              32   0.6520   0.6683     0.6748   0.7555     0.3041
A2 No Band Attn Within          32   0.6549   0.6672     0.6602   0.7432     0.2969
A3 Struct Only Within           32   0.6560   0.6536     0.6697   0.7470     0.2940
A4 No Hjorth Within             32   0.6534   0.6409     0.6611   0.7266    

In [2]:
import os, json
import numpy as np
from scipy.stats import wilcoxon

# ── Paths ──────────────────────────────────────────────────
BASE = "/kaggle/input/datasets/jerinanan/pgcn-all-results/all_results/results_eeg"

# ── Load per-subject results from a folder ─────────────────
def load_subjects(path, is_loso=False):
    results = {}
    if not os.path.exists(path):
        print(f"  ✗ Not found: {path}")
        return results
    prefix = "fold_" if is_loso else "s"
    for fd in sorted(os.scandir(path), key=lambda x: x.name):
        rpath = f"{fd.path}/metrics.json"
        if fd.name.startswith(prefix) and os.path.exists(rpath):
            key = fd.name.replace("fold_", "")
            results[key] = json.load(open(rpath))
    return results

# ── Wilcoxon test between two paired dicts ─────────────────
def wilcoxon_pair(res_a, res_b, task):
    shared = sorted(set(res_a.keys()) & set(res_b.keys()))
    if len(shared) < 8:
        return None
    a = np.array([res_a[s][task] for s in shared])
    b = np.array([res_b[s][task] for s in shared])
    diff = a - b
    if np.all(diff == 0):
        return {"n": len(shared), "delta": 0.0, "p": 1.0, "sig": "ns"}
    try:
        stat, p = wilcoxon(a, b, alternative="two-sided")
        r = np.sqrt(stat / (len(shared) * (len(shared) + 1) / 2))
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        return {"n": len(shared), "delta": float(np.mean(a) - np.mean(b)),
                "p": float(p), "r": float(r), "sig": sig}
    except Exception as e:
        return {"error": str(e)}

# ── Define experiments ──────────────────────────────────────
tasks = [("val_f1", "Valence"), ("aro_f1", "Arousal"),
         ("dom_f1", "Dominance"), ("lik_f1", "Liking"),
         ("quad_f1", "Quadrant")]

ablations = [
    ("A1: Flat GCN",    "within_flat_gcn",    "loso_flat_gcn"),
    ("A2: No Band Attn","within_no_band_attn", "loso_no_band"),      # ← changed
    ("A3: Struct Only", "within_struct_only",  "loso_struct_only"),
    ("A4: No Hjorth",   "within_no_hjorth",    "loso_no_hjorth"),
]

# ── Run for both protocols ──────────────────────────────────
for protocol, is_loso in [("Within-Subject", False), ("LOSO", True)]:
    prefix  = "loso_" if is_loso else "within_"
    full_dir = f"{BASE}/{prefix}none"
    full_res = load_subjects(full_dir, is_loso)
    n_full   = len(full_res)

    print(f"\n{'='*70}")
    print(f"  WILCOXON: Full PGCN vs Ablations  [{protocol}]  (n={n_full})")
    print(f"{'='*70}")
    print(f"  {'Ablation':<20} {'Task':<12} {'N':>3} {'Delta':>8} {'p':>8} {'r':>6}  Sig")
    print(f"  {'-'*65}")

    summary = {}   # collect for CSV

    for abl_name, within_dir, loso_dir in ablations:
        abl_dir = f"{BASE}/{loso_dir if is_loso else within_dir}"
        abl_res = load_subjects(abl_dir, is_loso)

        for task_key, task_label in tasks:
            result = wilcoxon_pair(full_res, abl_res, task_key)
            if result is None:
                print(f"  {abl_name:<20} {task_label:<12} {'<8':>3} {'skip':>8}")
                continue
            if "error" in result:
                print(f"  {abl_name:<20} {task_label:<12} ERROR: {result['error']}")
                continue

            delta_str = f"{result['delta']:+.4f}"
            p_str     = f"{result['p']:.4f}" if result['p'] >= 0.0001 else "<.0001"
            r_str     = f"{result.get('r', 0):.3f}"
            sig       = result['sig']

            # mark if full PGCN is BETTER (positive delta) or WORSE (negative)
            direction = "PGCN>" if result['delta'] > 0 else "ABL> "
            print(f"  {abl_name:<20} {task_label:<12} "
                  f"{result['n']:>3} {delta_str:>8} {p_str:>8} {r_str:>6}  "
                  f"{sig:<4} {direction}")

            summary[(abl_name, task_label)] = result

    print()

# ── Save to CSV ─────────────────────────────────────────────
import pandas as pd, os

rows = []
for protocol, is_loso in [("Within", False), ("LOSO", True)]:
    prefix  = "loso_" if is_loso else "within_"
    full_res = load_subjects(f"{BASE}/{prefix}none", is_loso)

    for abl_name, within_dir, loso_dir in ablations:
        abl_dir = f"{BASE}/{loso_dir if is_loso else within_dir}"
        abl_res = load_subjects(abl_dir, is_loso)
        for task_key, task_label in tasks:
            r = wilcoxon_pair(full_res, abl_res, task_key)
            if r and "error" not in r:
                rows.append({
                    "protocol": protocol,
                    "ablation": abl_name,
                    "task": task_label,
                    "n": r["n"],
                    "delta": round(r["delta"], 4),
                    "p": round(r["p"], 4),
                    "effect_r": round(r.get("r", 0), 3),
                    "sig": r["sig"]
                })

df = pd.DataFrame(rows)
os.makedirs("/kaggle/working", exist_ok=True)
df.to_csv("/kaggle/working/wilcoxon_ablation.csv", index=False)
print("✔ Saved to /kaggle/working/wilcoxon_ablation.csv")
print(df.to_string(index=False))


  WILCOXON: Full PGCN vs Ablations  [Within-Subject]  (n=32)
  Ablation             Task           N    Delta        p      r  Sig
  -----------------------------------------------------------------
  A1: Flat GCN         Valence       32  -0.0331   0.0945  0.574  ns   ABL> 
  A1: Flat GCN         Arousal       32  -0.0014   0.9099  0.656  ns   ABL> 
  A1: Flat GCN         Dominance     32  -0.0096   0.5387  0.577  ns   ABL> 
  A1: Flat GCN         Liking        32  -0.0256   0.2645  0.540  ns   ABL> 
  A1: Flat GCN         Quadrant      32  -0.0220   0.2169  0.611  ns   ABL> 
  A2: No Band Attn     Valence       32  -0.0360   0.3600  0.597  ns   ABL> 
  A2: No Band Attn     Arousal       32  -0.0003   0.9808  0.597  ns   ABL> 
  A2: No Band Attn     Dominance     32  +0.0050   0.6488  0.589  ns   PGCN>
  A2: No Band Attn     Liking        32  -0.0132   0.1329  0.509  ns   ABL> 
  A2: No Band Attn     Quadrant      32  -0.0148   0.4217  0.626  ns   ABL> 
  A3: Struct Only      Valence

In [1]:
import os, json, numpy as np, pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────
BASE     = "/kaggle/input/datasets/jerinanan/pgcn-all-results/all_results/results_eeg"
WITHIN   = f"{BASE}/within_none"

# The 22 subjects that have genuine facial video (same as fusion experiments)
FUSION_SUBJECTS = [f"s{i:02d}" for i in range(1, 23)]

# ── Load within-subject results for matched 22 subjects ────────────────
results_22 = {}
results_32 = {}

for fd in sorted(os.scandir(WITHIN), key=lambda x: x.name):
    rpath = f"{fd.path}/metrics.json"
    if fd.name.startswith("s") and os.path.exists(rpath):
        subj = fd.name
        m = json.load(open(rpath))
        results_32[subj] = m
        if subj in FUSION_SUBJECTS:
            results_22[subj] = m

print(f"Within-subject all 32 subjects : {len(results_32)}")
print(f"Within-subject matched 22 subjects: {len(results_22)}")
print(f"Matched subjects: {sorted(results_22.keys())}")

# ── Compute summary statistics ─────────────────────────────────────────
tasks = [
    ("val_f1",  "Valence"),
    ("aro_f1",  "Arousal"),
    ("dom_f1",  "Dominance"),
    ("lik_f1",  "Liking"),
    ("quad_f1", "Quadrant"),
]

def summarise(results, label):
    print(f"\n{'='*60}")
    print(f"  EEG-Only Within-Subject — {label}  (n={len(results)})")
    print("="*60)
    for task_key, task_label in tasks:
        vals = [v[task_key] for v in results.values() if task_key in v]
        if vals:
            print(f"  {task_label:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}"
                  f"  (min={np.min(vals):.4f}  max={np.max(vals):.4f})")

summarise(results_32, "all 32 subjects")
summarise(results_22, "matched 22 subjects")

# ── Per-subject table for the 22 subjects ─────────────────────────────
print(f"\n{'='*60}")
print("  PER-SUBJECT — matched 22 subjects")
print("="*60)
print(f"  {'Subj':>5} {'Val':>8} {'Aro':>8} {'Dom':>8} {'Lik':>8} {'Quad':>8}")
print(f"  {'-'*50}")
for subj in sorted(results_22.keys()):
    m = results_22[subj]
    print(f"  {subj:>5} "
          f"{m.get('val_f1',0):>8.4f} "
          f"{m.get('aro_f1',0):>8.4f} "
          f"{m.get('dom_f1',0):>8.4f} "
          f"{m.get('lik_f1',0):>8.4f} "
          f"{m.get('quad_f1',0):>8.4f}")

# ── Save both ──────────────────────────────────────────────────────────
out_dir = "/kaggle/working/eeg_within_matched"
os.makedirs(out_dir, exist_ok=True)

# Per-subject CSV for matched 22
df_22 = pd.DataFrame(results_22).T
df_22.index.name = "subject"
df_22.to_csv(f"{out_dir}/per_subject_22.csv", float_format="%.4f")

# Per-subject CSV for all 32
df_32 = pd.DataFrame(results_32).T
df_32.index.name = "subject"
df_32.to_csv(f"{out_dir}/per_subject_32.csv", float_format="%.4f")

# Global summary JSON for matched 22
summary_22 = {}
for task_key, task_label in tasks:
    vals = [v[task_key] for v in results_22.values() if task_key in v]
    if vals:
        summary_22[f"{task_key}_mean"] = round(float(np.mean(vals)), 4)
        summary_22[f"{task_key}_std"]  = round(float(np.std(vals)),  4)
        summary_22[f"{task_key}_min"]  = round(float(np.min(vals)),  4)
        summary_22[f"{task_key}_max"]  = round(float(np.max(vals)),  4)

json.dump(summary_22, open(f"{out_dir}/global_metrics_22.json", "w"), indent=2)

# Global summary JSON for all 32
summary_32 = {}
for task_key, task_label in tasks:
    vals = [v[task_key] for v in results_32.values() if task_key in v]
    if vals:
        summary_32[f"{task_key}_mean"] = round(float(np.mean(vals)), 4)
        summary_32[f"{task_key}_std"]  = round(float(np.std(vals)),  4)

json.dump(summary_32, open(f"{out_dir}/global_metrics_32.json", "w"), indent=2)

print(f"\n✔ Saved to {out_dir}/")
print(f"  per_subject_22.csv")
print(f"  per_subject_32.csv")
print(f"  global_metrics_22.json")
print(f"  global_metrics_32.json")

# ── Zip for download ───────────────────────────────────────────────────
import shutil
shutil.make_archive(
    "/kaggle/working/eeg_within_matched",
    "zip",
    "/kaggle/working",
    "eeg_within_matched"
)
print(f"\n✔ {os.path.getsize('/kaggle/working/eeg_within_matched.zip')/1e6:.2f} MB")

Within-subject all 32 subjects : 32
Within-subject matched 22 subjects: 22
Matched subjects: ['s01', 's02', 's03', 's04', 's05', 's06', 's07', 's08', 's09', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 's22']

  EEG-Only Within-Subject — all 32 subjects  (n=32)
  Valence     : 0.6189 ± 0.1891  (min=0.1974  max=0.9091)
  Arousal     : 0.6669 ± 0.2051  (min=0.0385  max=0.9091)
  Dominance   : 0.6652 ± 0.2399  (min=0.1159  max=1.0000)
  Liking      : 0.7300 ± 0.1651  (min=0.4413  max=1.0000)
  Quadrant    : 0.2821 ± 0.1214  (min=0.1236  max=0.6890)

  EEG-Only Within-Subject — matched 22 subjects  (n=22)
  Valence     : 0.5968 ± 0.1795  (min=0.2016  max=0.8263)
  Arousal     : 0.6579 ± 0.2278  (min=0.0385  max=0.9091)
  Dominance   : 0.5987 ± 0.2444  (min=0.1159  max=0.9381)
  Liking      : 0.7499 ± 0.1723  (min=0.4413  max=1.0000)
  Quadrant    : 0.2588 ± 0.1033  (min=0.1236  max=0.4879)

  PER-SUBJECT — matched 22 subjects
   Subj      Val      Aro

In [8]:
import os, json, numpy as np, pandas as pd
from pathlib import Path
from scipy import stats

# ── SET YOUR PATH HERE ──
RESULTS = "/kaggle/input/datasets/jerinanan/pgcn-all-results/all_results"
OUT     = "/kaggle/working/corrected_statistics_all_comparisons.csv"

# ── Everything below is identical to the script — no changes needed ──
TASKS = ["val", "aro", "dom", "lik", "quad"]
TL = {"val":"Valence","aro":"Arousal","dom":"Dominance","lik":"Liking","quad":"Quadrant"}

def load(root, exp_path):
    base = Path(root) / exp_path; out = {}
    for sd in sorted(base.glob("s*")):
        mp = sd/"metrics.json"
        if mp.exists(): out[sd.name] = json.load(open(mp))
    for fd in sorted(base.glob("fold_s*")):
        mp = fd/"metrics.json"
        if mp.exists(): out[fd.name.replace("fold_","")] = json.load(open(mp))
    return out

def paired(A, B, task):
    common = sorted(set(A)&set(B))
    return (np.array([A[s][f"{task}_f1"] for s in common]),
            np.array([B[s][f"{task}_f1"] for s in common]))

def wilcoxon_full(a, b):
    d = a-b; nz = d!=0; d = d[nz]; n = len(d)
    if n==0: return dict(n=0,p=1.0,r_rb=0.0,delta=float(np.mean(a-b)))
    ranks = stats.rankdata(np.abs(d))
    Wp = ranks[d>0].sum(); Wm = ranks[d<0].sum(); T = n*(n+1)/2
    r_rb = (Wp-Wm)/T
    try: _,p = stats.wilcoxon(a,b,zero_method="wilcox",alternative="two-sided")
    except ValueError: p=1.0
    return dict(n=n,p=float(p),r_rb=float(r_rb),delta=float(np.mean(a-b)))

def holm(ps):
    ps=np.asarray(ps); m=len(ps); order=np.argsort(ps)
    adj=np.empty(m); run=0.0
    for k,i in enumerate(order): run=max(run,(m-k)*ps[i]); adj[i]=min(run,1.0)
    return adj

def bh(ps):
    ps=np.asarray(ps); m=len(ps); order=np.argsort(ps)
    adj=np.empty(m); run=1.0
    for k in range(m-1,-1,-1):
        i=order[k]; run=min(run,ps[i]*m/(k+1)); adj[i]=min(run,1.0)
    return adj

def sig(p): return "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else "ns"

def run_family(title, comparisons):
    rows=[]
    for task,(la,A),(lb,B) in comparisons:
        a,b = paired(A,B,task); r=wilcoxon_full(a,b)
        rows.append(dict(Family=title,Task=TL[task],
                         Comparison=f"{la} vs {lb}",n=r["n"],
                         mean_diff=round(r["delta"],4),
                         r_rankbiserial=round(r["r_rb"],3),
                         p_raw=round(r["p"],4)))
    df=pd.DataFrame(rows)
    df["p_Holm"]=np.round(holm(df["p_raw"].values),4)
    df["p_BH"]  =np.round(bh(df["p_raw"].values),4)
    df["sig_raw"]=df["p_raw"].map(sig)
    df["sig_Holm"]=df["p_Holm"].map(sig)
    df["sig_BH"]=df["p_BH"].map(sig)
    return df

# ── Load ──
EEG = Path(RESULTS)/"results_eeg"
FUS = Path(RESULTS)/"results_fusion"

print("Checking paths:")
print(f"  EEG    exists: {EEG.exists()}")
print(f"  FUSION exists: {FUS.exists()}")
if EEG.exists(): print(f"  EEG contents: {[d.name for d in EEG.iterdir()]}")
if FUS.exists(): print(f"  FUSION contents: {[d.name for d in FUS.iterdir()]}")

W_eeg  = load(EEG,"within_none");   L_eeg  = load(EEG,"loso_none")
W_early= load(FUS,"within_early");  W_late = load(FUS,"within_late")
L_early= load(FUS,"loso_early");    L_late = load(FUS,"loso_late")

abl_w={"A1 FlatGCN":   load(EEG,"within_flat_gcn"),
        "A2 NoBandAttn":load(EEG,"within_no_band_attn"),
        "A3 StructOnly":load(EEG,"within_struct_only"),
        "A4 NoHjorth":  load(EEG,"within_no_hjorth")}
abl_l={"A1 FlatGCN":   load(EEG,"loso_flat_gcn"),
        "A2 NoBandAttn":load(EEG,"loso_no_band"),   # folder is loso_no_band not loso_no_band_attn
        "A3 StructOnly":load(EEG,"loso_struct_only"),
        "A4 NoHjorth":  load(EEG,"loso_no_hjorth")}

print(f"\nSubjects loaded — W_eeg:{len(W_eeg)} L_eeg:{len(L_eeg)} "
      f"W_early:{len(W_early)} W_late:{len(W_late)} "
      f"L_early:{len(L_early)} L_late:{len(L_late)}")
for k,v in abl_w.items(): print(f"  within {k}: {len(v)}")
for k,v in abl_l.items(): print(f"  loso   {k}: {len(v)}")

# ── Run families ──
families=[]
comp=[]
for t in TASKS:
    comp+=[(t,("EEG-Only",W_eeg),("Early",W_early)),
           (t,("EEG-Only",W_eeg),("Late", W_late)),
           (t,("Early",W_early), ("Late", W_late))]
families.append(run_family("Within-fusion",comp))

comp=[]
for t in TASKS:
    comp+=[(t,("EEG-Only",L_eeg),("Early",L_early)),
           (t,("EEG-Only",L_eeg),("Late", L_late)),
           (t,("Early",L_early), ("Late", L_late))]
families.append(run_family("LOSO-fusion",comp))

comp=[]
for t in TASKS:
    for k,A in abl_w.items(): comp.append((t,("Full",W_eeg),(k,A)))
families.append(run_family("Ablation-within",comp))

comp=[]
for t in TASKS:
    for k,A in abl_l.items(): comp.append((t,("Full",L_eeg),(k,A)))
families.append(run_family("Ablation-LOSO",comp))

# ── Output ──
out=pd.concat(families,ignore_index=True)[
    ["Family","Task","Comparison","n","mean_diff",
     "r_rankbiserial","p_raw","sig_raw","p_Holm","sig_Holm","p_BH","sig_BH"]]
out.to_csv(OUT,index=False)
print(f"\nSaved {len(out)} comparisons → {OUT}")
print(out.to_string(index=False))

Checking paths:
  EEG    exists: True
  FUSION exists: True
  EEG contents: ['loso_no_hjorth', 'loso_flat_gcn', 'loso_no_band', 'within_struct_only', 'within_none', 'within_flat_gcn', 'within_no_band_attn', 'loso_struct_only', 'loso_none', 'within_no_hjorth']
  FUSION contents: ['loso_late', 'within_late', 'loso_early', 'within_early']

Subjects loaded — W_eeg:32 L_eeg:32 W_early:22 W_late:22 L_early:22 L_late:22
  within A1 FlatGCN: 32
  within A2 NoBandAttn: 32
  within A3 StructOnly: 32
  within A4 NoHjorth: 32
  loso   A1 FlatGCN: 32
  loso   A2 NoBandAttn: 32
  loso   A3 StructOnly: 32
  loso   A4 NoHjorth: 32

Saved 70 comparisons → /kaggle/working/corrected_statistics_all_comparisons.csv
         Family      Task            Comparison  n  mean_diff  r_rankbiserial  p_raw sig_raw  p_Holm sig_Holm   p_BH sig_BH
  Within-fusion   Valence     EEG-Only vs Early 21    -0.1113          -0.455 0.0680      ns  0.8088       ns 0.2040     ns
  Within-fusion   Valence      EEG-Only vs Late 